In [ ]:
from pathlib import Path

PDF_PATH = "36374021.pdf" 

In [ ]:
%pip install pymupdf

import fitz

doc = fitz.open(PDF_PATH)

text = ""
for page in doc:
    text += page.get_text()

print(text)

In [ ]:
# See token counts
# %pip install transformers
from transformers import AutoTokenizer

model_name = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokens = tokenizer.encode(text)

print("Token count:", len(tokens))

In [ ]:
# %pip install chonkie
from chonkie import RecursiveChunker
from chonkie.refinery import OverlapRefinery

chunker = RecursiveChunker(
    chunk_size=512
)
chunks = chunker.chunk(text)

refinery = OverlapRefinery(
    context_size=50
)
ol_chunks = refinery.refine(chunks)

# Print the results
print(f"Refined Chunks: {len(ol_chunks)}")
# for i, chunk in enumerate(ol_chunks):
#     print(f"\n--- Chunk {i+1} ---")
#     print(chunk.text.strip())
#     print(f"Tokens: {chunk.token_count}")


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio"
)

embeddings_list = []
for i, text in enumerate(ol_chunks):
    # Send the text to the LM Studio API
    response = client.embeddings.create(
        input=text,
        model="text-embedding-nomic-embed-text-v1.5"
    )
    # Extract the vector from the response
    embeddings_list.append(response.data[0].embedding)
    
    progress = (i + 1) / len(ol_chunks) * 100
    print(f"Progress: {int(progress)}% - Processed chunk {i + 1} of {len(ol_chunks)}")

print("\nEmbedding process complete!")
print(f"Successfully created {len(embeddings_list)} embeddings.")

In [ ]:
import chromadb

COLLECTION_NAME = "rag_document_store"
PERSIST_DIRECTORY = "./chroma_db_data"

# 1. Initialize the Chroma client with a persistent directory
# This tells ChromaDB where to store all its files on disk.
client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)

# 2. Create or get the collection
collection = client.get_or_create_collection(COLLECTION_NAME)


# --- Data Preparation and Extraction Step (Using previous logic) ---
ids = []
documents = []
embeddings = embeddings_list # Assuming this list is available

# Check if the item is a valid Chunk object and extract its text
for i, chunk in enumerate(ol_chunks):
    document_text = chunk.text.strip()
    documents.append(document_text)
    ids.append(f"chunk_{i}")

# 4. Add the documents, embeddings, and IDs to the collection (This writes to disk)
if documents:
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )
    print(f"Successfully indexed {len(documents)} clean chunks into ChromaDB collection: {COLLECTION_NAME}")
else:
    print("No valid text content was extracted to index.")

In [ ]:
from openai import OpenAI

# Initialize the client (assuming it's already set up as in your previous code)
client = OpenAI(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio"
)

user_query = "What is the main bacteria studied in the article"

# Embed the user query
response = client.embeddings.create(
    input=user_query,
    model="text-embedding-nomic-embed-text-v1.5"
)

# Extract the vector from the response
query_embedding = response.data[0].embedding

In [ ]:
import chromadb

# --- Configuration based on previous steps ---
PERSIST_DIRECTORY = "./chroma_db_data"
COLLECTION_NAME = "rag_document_store"

# 1. Initialize the Chroma client pointing to the persistent data directory
client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)

# 2. Get the collection from the loaded client
collection = client.get_collection(COLLECTION_NAME)

print(f"Searching for context using query: '{user_query}'...")

# 3. Perform the similarity search (Retrieval)
results = collection.query(
    query_embeddings=[query_embedding],  
    n_results=3,
)

In [ ]:
for i in range(len(results['ids'][0])):
    doc_id = results['ids'][0][i]
    doc_text = results['documents'][0][i]

    print(f"{doc_id}:")
    print(doc_text)
    print("=" * 40)

### You are a expart data extractor. you extract key information from research articals about bacterio phage and help create dataset. Extract the following information from the given research article:
* PMID:
* Targeted bacteria:
* Bacterial Strain/isolate:	
* Phage:
* Place of Sample collection:
* Phage isolation Sample:
* Phage Plaque characteristics:
* Phage TEM morphology:
* Phage TEM dimensions:
* Phage Taxonomy:
* Phage type (Lytic/ Lysogenic/ Engineered):
* Optimal MOI:
* Latent period (min):
* Burst size (phage/infected bacterium):
* Optimal Temperature (°C):
* Optimal pH:
* Phage Genome size (bp):
* Phage GC content(%):
* Phage Genome Accession/Bioproject:

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
)

response = client.responses.create(
    model="google/gemma-4-e2b",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)